In [1]:
from dotenv import load_dotenv 
import os 

load_dotenv()

True

In [ ]:
import os 

DB_URI = os.getenv("DB_URI")

In [ ]:
#@markdown Check Table

import psycopg2

DB_URI = DB_URI

conn = psycopg2.connect(DB_URI)
cur = conn.cursor()

cur.execute("""
    SELECT table_name 
    FROM information_schema.tables
    WHERE table_schema = 'public';
""")

tables = cur.fetchall()

print("Tables:")
for table in tables:
    print(table[0])

cur.close()
conn.close()

Tables:
checkpoint_migrations
checkpoints
checkpoint_blobs
checkpoint_writes


In [ ]:
#@markdown Delete Tables
import psycopg

DB_URI = DB_URI

with psycopg.connect(DB_URI, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            DO $$ DECLARE
                r RECORD;
            BEGIN
                -- Drop all tables
                FOR r IN (SELECT tablename FROM pg_tables WHERE schemaname = 'public') LOOP
                    EXECUTE 'DROP TABLE IF EXISTS ' || quote_ident(r.tablename) || ' CASCADE';
                END LOOP;
            END $$;
        """)
        print("All tables deleted successfully.")

All tables deleted successfully.


In [ ]:
from sqlalchemy import create_engine, inspect, text

DB_URI = DB_URI

engine = create_engine(DB_URI)
inspector = inspect(engine)

tables = inspector.get_table_names()

for table in tables:
    print(f"\n===== TABLE: {table} =====")
    
    with engine.connect() as conn:
        result = conn.execute(text(f"SELECT * FROM {table} LIMIT 5"))
        rows = result.fetchall()
        
        if rows:
            for row in rows:
                print(row)
        else:
            print("No data.")


===== TABLE: checkpoint_writes =====
('share_thread', '', '1f11a3c5-7faf-6bd6-bfff-88135c6a750d', 'ee65eb3d-dc68-7377-7bd1-c29b617f5c2b', 0, 'messages', 'msgpack', <memory at 0x116091240>, '~__pregel_pull, __start__')
('share_thread', '', '1f11a3c5-7faf-6bd6-bfff-88135c6a750d', 'ee65eb3d-dc68-7377-7bd1-c29b617f5c2b', 1, 'branch:to:model', 'null', <memory at 0x116091300>, '~__pregel_pull, __start__')
('share_thread', '', '1f11a3c5-7fb1-60a8-8000-25529793692d', '26bb1980-d0ea-d3aa-280a-a196d50fcd32', 0, 'messages', 'msgpack', <memory at 0x1160913c0>, '~__pregel_pull, model')

===== TABLE: checkpoint_migrations =====
(0,)
(1,)
(2,)
(3,)
(4,)

===== TABLE: checkpoints =====
('share_thread', '', '1f11a3c5-7faf-6bd6-bfff-88135c6a750d', None, None, {'v': 4, 'id': '1f11a3c5-7faf-6bd6-bfff-88135c6a750d', 'ts': '2026-03-07T15:43:10.162220+00:00', 'versions_seen': {'__input__': {}}, 'channel_values': {}, 'channel_versions': {'__start__': '00000000000000000000000000000001.0.5236310547938007'}, 'u

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    
    agent2_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_2"
    ]
    

    agent3_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_3_as_imposter"
    ]
    
    #print("Agent2:", agent2_words)

    #print(request.messages)

   # print("Latest", words)

    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are a player in an Imposter social deduction game.
                    Players' words so far: {words}
                    Your secret keyword is {content}: 

                    Rules:
                    - Answer in ONE word only.
                    - Directly describe what it is or how it is used.
                    - Do NOT repeat previous words from other players.
                    - Be subtle. Do not make it too obvious.
                    """

    elif round_c in ["round_2", "round_3"]:
        return f"""
                Players' words so far: {words}
                Your keyword: {content}
                Think carefully:
                - Then give ONE new word.
                - Do NOT repeat any previous word.
                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_2's responses:
            {agent2_words}

            Player_3's responses:
            {agent3_words}

            Based on these responses:
            - Guess who is the imposter (Player_2 or Player_3).
            - Return only the player name.
            """
    else:
        pass 
    return round_c
DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_1",
        checkpointer=checkpointer,
    )
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Ronaldo"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    

    agent1_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_1"
    ]

    agent3_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_3_as_imposter"
    ]
    
    #print("Agent1:", agent1_words)

    #print(request.messages)

   # print("Latest", words)

    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are a player in an Imposter social deduction game.
                    Players' words so far: {words}
                    Your secret keyword is {content}: 

                    Rules:
                    - Answer in ONE word only.
                    - Directly describe what it is or how it is used.
                    - Do NOT repeat previous words from other players.
                    - Be subtle. Do not make it too obvious.

                    """

    elif round_c in ["round_2", "round_3"]:
        return f"""
                Players' words so far: {words}
                Your keyword: {content}
                Think carefully:
                - Then give ONE new word.
                - Do NOT repeat any previous word.

                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_1's responses:
            {agent1_words}

            Player_3's responses:
            {agent3_words}


            Based on these responses:
            - Guess who is the imposter (Player_1 or Player_3).
            - Return only the player name.
            """
    else:
        pass 
    return round_c
DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_2",
        checkpointer=checkpointer,
    )
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Ronaldo"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.postgres import PostgresSaver  
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b",reasoning_format="hidden")

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def round_prompt(request: ModelRequest) -> str:
    round_c = request.runtime.context.get("count_round", "round_c")
    content = request.messages[0].content
    #print(content)
    
    words = [
        msg.content
        for msg in request.messages
        if isinstance(msg, AIMessage)
    ]
    
   # print("Latest", words)
    agent1_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_1"
    ]

    agent2_words = [
        msg.content for msg in request.messages
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "agent_2"
    ]


    words = ", ".join(words) if words else "None yet"
    
    if round_c == "round_1":
        return f"""You are an Imposter in a social deduction game.
            Players' words so far: {words}

            You DO NOT know the secret word.

            Rules:
            - Carefully analyze other players' responses.
            - Give ONE word only.
            - Do NOT repeat previous words.
            - Blend in naturally.
            - Try to guess what the keyword might be.
            - In Burmese

"""

    elif round_c in ["round_2", "round_3"]:
        return f"""
        Players' words so far:
        {words}

        You are the Imposter.

        - Analyze all clues.
        - Guess what the secret word might be.
        - Blend in naturally.
        - Give ONE new word.
        - Do NOT repeat previous words.
        - In Burmese

                """

    elif round_c in ['guess_imposter']:
        return f"""
            Player_1's responses:
            {agent1_words}

            Player_2's responses:
            {agent2_words}


            Based on these responses:
            - Guess who is the imposter (Player_1 or Player_2).
            - Return only the player name.
            """
    else:
        pass 
    return round_c
DB_URI = DB_URI

with PostgresSaver.from_conn_string(DB_URI) as checkpointer:
    checkpointer.setup() # auto create tables in PostgresSql
    imposter_agent = create_agent(
        model=model,
        middleware=[round_prompt],
        context_schema=Context,
        name="agent_3_as_imposter",
        checkpointer=checkpointer,
    )
    
    imposter_agent_rt = imposter_agent.invoke(
        {"messages": [{"role": "user", "content": "you are an imposter"}]},
        context={"count_round": "guess_imposter"}, 
        config= {"configurable": {"thread_id": "share_thread"}})
    

In [21]:
result['messages'][1].content

'ပန်း'

In [ ]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent1 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "share_thread"
    }
}

agent1_history = checkpointer_agent1.get_tuple(config)
agent1_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f11a1b0-f517-6674-8010-8784a1f54704'}}, checkpoint={'v': 4, 'id': '1f11a1b0-f517-6674-8010-8784a1f54704', 'ts': '2026-03-07T11:44:54.863203+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000017.0.49542560440222816'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000016.0.06277355478253399'}}, 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='8ecf016d-919c-4c8e-836c-96eacc124dfb'), AIMessage(content='Dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 181, 'prompt_tokens': 149, 'total_tokens': 330, 'completion_time': 0.390330301, 'completion_tokens_details': {'reasoning_tokens': 170}, 'prompt_time': 0.006795929, 'prompt_tokens_details': None, 'queue_time': 0.044870721, 'total_time': 0.39712623}, 'model_name': 'openai/gpt-oss-

In [ ]:
import psycopg

conn = psycopg.connect(DB_URI)

checkpointer_agent2 = PostgresSaver(conn)

config = {
    "configurable": {
        "thread_id": "2"
    }
}

agent2_history = checkpointer_agent2.get_tuple(config)
agent2_history

In [23]:
agent2_history[1]

{'v': 4,
 'id': '1f119731-5792-63a6-8001-1b08232782ca',
 'ts': '2026-03-06T15:42:29.738985+00:00',
 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000002.0.5432758134985926'},
  '__input__': {},
  '__start__': {'__start__': '00000000000000000000000000000001.0.684758239359927'}},
 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='14e575df-038b-4bbe-b0a0-e5b59a7f8f66'),
   AIMessage(content='treat', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 236, 'prompt_tokens': 149, 'total_tokens': 385, 'completion_time': 0.504629861, 'completion_tokens_details': {'reasoning_tokens': 225}, 'prompt_time': 0.005667934, 'prompt_tokens_details': None, 'queue_time': 0.045236996, 'total_time': 0.510297795}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='age

In [12]:
agent1_history

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-4df2-6cca-8010-97e64aa504cf'}}, checkpoint={'v': 4, 'id': '1f119811-4df2-6cca-8010-97e64aa504cf', 'ts': '2026-03-06T17:22:41.684176+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000017.0.09720644422819291'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000016.0.1913759363839982'}}, 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='413be9ad-d00c-4980-ab2b-628f55b03809'), AIMessage(content='dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 149, 'total_tokens': 338, 'completion_time': 0.411281686, 'completion_tokens_details': {'reasoning_tokens': 178}, 'prompt_time': 0.006028855, 'prompt_tokens_details': None, 'queue_time': 0.044320374, 'total_time': 0.417310541}, 'model_name': 'openai/gpt-oss-

In [9]:
agent1 = []
agent2 = []
agent_3_as_imposter = []

for take_name_agent in agent1_history[1]['channel_values']['messages']:
    if take_name_agent.name == "agent_1":
        agent1.append(take_name_agent.content)
    elif take_name_agent.name == "agent_2":
        agent2.append(take_name_agent.content)
    elif take_name_agent.name == "agent_3_as_imposter":
        agent_3_as_imposter.append(take_name_agent.content)
    else:
        pass
print("Agent1:" , agent1)
print("Agent2:" , agent2)
print("Agent3:", agent_3_as_imposter)

Agent1: ['Dessert', 'Muffin']
Agent2: ['Pastry', 'Bread']
Agent3: ['Cake', 'Cookie']


In [29]:
agent1_history[1]['channel_values']['messages']

[HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='413be9ad-d00c-4980-ab2b-628f55b03809'),
 AIMessage(content='dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 149, 'total_tokens': 338, 'completion_time': 0.411281686, 'completion_tokens_details': {'reasoning_tokens': 178}, 'prompt_time': 0.006028855, 'prompt_tokens_details': None, 'queue_time': 0.044320374, 'total_time': 0.417310541}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-13bf-7b43-b929-5e3db6ed3026-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 189, 'total_tokens': 338, 'output_token_details': {'reasoning': 178}}),
 HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='f1fb0093-30e7-4dc6-aee7-a2e269984af3

CheckpointTuple(config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-4df2-6cca-8010-97e64aa504cf'}}, checkpoint={'v': 4, 'id': '1f119811-4df2-6cca-8010-97e64aa504cf', 'ts': '2026-03-06T17:22:41.684176+00:00', 'versions_seen': {'model': {'branch:to:model': '00000000000000000000000000000017.0.09720644422819291'}, '__input__': {}, '__start__': {'__start__': '00000000000000000000000000000016.0.1913759363839982'}}, 'channel_values': {'messages': [HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='413be9ad-d00c-4980-ab2b-628f55b03809'), AIMessage(content='dessert', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 149, 'total_tokens': 338, 'completion_time': 0.411281686, 'completion_tokens_details': {'reasoning_tokens': 178}, 'prompt_time': 0.006028855, 'prompt_tokens_details': None, 'queue_time': 0.044320374, 'total_time': 0.417310541}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-13bf-7b43-b929-5e3db6ed3026-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 189, 'total_tokens': 338, 'output_token_details': {'reasoning': 178}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='f1fb0093-30e7-4dc6-aee7-a2e269984af3'), AIMessage(content='treat', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 227, 'prompt_tokens': 165, 'total_tokens': 392, 'completion_time': 0.479108877, 'completion_tokens_details': {'reasoning_tokens': 216}, 'prompt_time': 0.006323071, 'prompt_tokens_details': None, 'queue_time': 0.046116659, 'total_time': 0.485431948}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-16a1-7891-a646-0fcb36c38a17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 227, 'total_tokens': 392, 'output_token_details': {'reasoning': 216}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='3b399e81-7d8c-49ea-a609-b81a112ff08e'), AIMessage(content='cake', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 231, 'prompt_tokens': 152, 'total_tokens': 383, 'completion_time': 0.500922681, 'completion_tokens_details': {'reasoning_tokens': 221}, 'prompt_time': 0.005771719, 'prompt_tokens_details': None, 'queue_time': 0.04583249, 'total_time': 0.5066944}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-298d-70c0-9d2f-1761c490f878-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 231, 'total_tokens': 383, 'output_token_details': {'reasoning': 221}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='c8614676-da6d-4eed-a173-26a0db3d898b'), AIMessage(content='pastry', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 169, 'total_tokens': 323, 'completion_time': 0.318490351, 'completion_tokens_details': {'reasoning_tokens': 143}, 'prompt_time': 0.006637686, 'prompt_tokens_details': None, 'queue_time': 0.045191942, 'total_time': 0.325128037}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e10890e4b9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-2cbd-7d03-b1ac-78cc8ac9697c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 169, 'output_tokens': 154, 'total_tokens': 323, 'output_token_details': {'reasoning': 143}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='103284ff-b1a0-429c-a74c-dc89644579c4'), AIMessage(content='pie', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 187, 'total_tokens': 349, 'completion_time': 0.344637025, 'completion_tokens_details': {'reasoning_tokens': 152}, 'prompt_time': 0.007685024, 'prompt_tokens_details': None, 'queue_time': 0.045595085, 'total_time': 0.352322049}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_1', id='lc_run--019cc42c-40e9-7373-b0f7-086b5306c3f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 187, 'output_tokens': 162, 'total_tokens': 349, 'output_token_details': {'reasoning': 152}}), HumanMessage(content='မုန့်', additional_kwargs={}, response_metadata={}, id='c64a0b08-d3ce-43d2-b717-97c4e41cecca'), AIMessage(content='cookie', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 204, 'total_tokens': 372, 'completion_time': 0.354691989, 'completion_tokens_details': {'reasoning_tokens': 158}, 'prompt_time': 0.008408945, 'prompt_tokens_details': None, 'queue_time': 0.045283894, 'total_time': 0.363100934}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, name='agent_2', id='lc_run--019cc42c-4386-7b13-91de-79d611429c8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 204, 'output_tokens': 168, 'total_tokens': 372, 'output_token_details': {'reasoning': 158}})]}, 'channel_versions': {'messages': '00000000000000000000000000000018.0.9591710399123636', '__start__': '00000000000000000000000000000017.0.09720644422819291', 'branch:to:model': '00000000000000000000000000000018.0.9591710399123636'}, 'updated_channels': ['messages']}, metadata={'step': 16, 'source': 'loop', 'parents': {}, 'lc_agent_name': 'agent_2'}, parent_config={'configurable': {'thread_id': 'share_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f119811-48ea-680e-800f-67decfe182a4'}}, pending_writes=[])

In [6]:
from langchain_core.messages import AIMessage

ai_contents = [
    msg.content
    for msg in agent1_history[1]["channel_values"]["messages"]
    if isinstance(msg, AIMessage)
]

ai_contents

['dessert', 'treat', 'cake', 'pastry', 'pie', 'cookie']

In [19]:
from langchain_core.messages import AIMessage

ai_contents = [
    msg.content
    for msg in agent1_history[1]["channel_values"]["messages"]
    if isinstance(msg, AIMessage)
]

ai_contents

['dessert', 'treat', 'cake', 'pastry', 'pie', 'cookie']

In [ ]:
['pastry', 'dessert', 'baked', 'snack']


In [ ]:
['pastry', 'dessert', 'baked']
['treat', 'snack', 'pastry']

In [16]:
thread_id = "1"
checkpoint_deleter = checkpointer.delete_thread(thread_id)

In [17]:
checkpoint_deleter

In [18]:
checkpointer.get_tuple(config)